In [1]:
import os
import json
import random
from glob import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"-Device:{device}")
if torch.cuda.is_available():
    print(f" GPU:{torch.cuda.get_device_name(0)}")


-Device:cpu


In [2]:

DATA_ROOT = "/kaggle/input/xbd-dataset/xbd"
OUT_DIR = "/kaggle/working/index"
os.makedirs(OUT_DIR, exist_ok = True)
random.seed(SEED)
def index_split(split_name, ratio):
    base = os.path.join(DATA_ROOT, split_name)
    img_dir = os.path.join(base, "images")
    mask_dir = os.path.join (base, "maske")
    pre_images = sorted(glob(os.path.join(img_dir, "*_pre_disaster.png")))
    random.shuffle(pre_images)
    selected = pre_images [:int(len(pre_images) * ratio)]
    index = []
    for pre_path in tqdm(selected, desc=f"Indexing {split_name}"):
        base_name = base_name = os.path.basename(pre_path).replace("_pre_disaster.png", "")
        post_path = os.path.join(img_dir, base_name + "_post_disaster.png")
        mask_path = os.path.join(mask_dir, base_name + "_post_disaster.png")
        if os.path.exits(post_path) and os.path.exists(mask_path):
            index.append({
                "id": base_name,
                "pre": pre_path,
                "post": post_path,
                "mask": mask_path
            })
    return index
tier1_index = index_split("tier1", 0.75)
tier3_index = index_split("tier3", 0.70)
train_index = tier1_index + tier3_index
random.shuffle(train_index)
split = int (0.7 * len (train_index))
train_data = train_index[:split]
val_data = train_index[split:]
test_data = index_split("test", 0.40)
json.dump(train_data, open(f"{OUT_DIR}/train.json", "w"), indent=2)
json.dump(val_data, open(f"{OUT_DIR}/val.json", "w"), indent=2)
json.dump(test_data, open(f"{OUT_DIR}/test.json", "w"), indent=2)
print("\n✓ Indexing complete")
print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

Indexing tier1: 0it [00:00, ?it/s]
Indexing tier3: 0it [00:00, ?it/s]
Indexing test: 0it [00:00, ?it/s]


✓ Indexing complete
Train: 0
Val:   0
Test:  0
